In [1]:
import pandas as pd
import os as os
import spacy
import re
from tqdm.notebook import tqdm

In [2]:
def text_normalize(text):
    ''' 
    Returns text after doing the following : 
        1. Lowercases all alphabets
        2. Replaces all URLs with <URL> tag
        3. Replaces all numbers (2 or more adjacent) with <NUMBER> tag
        4. Replaces all character repetitions (more than 2) with 2 of same character
        5. Removes all punctuation
        6. Replaces all instances of 2 or more adjacent spaces with single space
        
    '''
    
    text = text.lower() 
    text = re.sub(r'(https?://|www\.)\S+', '<URLTOKEN>', text) 
    text = re.sub(r'[0-9]{2,}', '<NUMBERTOKEN>', text) 

    text = re.sub(r'(.)\1{2,}', '\1\1', text)
    text = re.sub(r'[^\s\w]', ' ', text)
    text = re.sub(r'\s+', ' ', text)

    
    return text.strip()
    

In [3]:
def lemmatize(all_texts, batch_size = 100):
    nlp = spacy.load('en_core_web_sm')
    lemmatized = []
    nlp_pipe = nlp.pipe(all_texts, disable = ['ner', 'parser'], batch_size = batch_size, n_process = 2)
    
    for doc in tqdm(nlp_pipe, desc = 'Processing text....'):
        all_lemmas = []
        for token in doc:
            all_lemmas.append(token.lemma_)

        text = ' '.join(all_lemmas)
        lemmatized.append(text)
    
    return lemmatized
        

In [4]:
def quality_check(df):
    df = df.drop_duplicates()
    df = df.dropna()
    df = df.reset_index()
    
    return df

In [5]:
def text_preprocessing(df):
    df['comment_text'] = df['comment_text'].apply(text_normalize)
    all_texts, batch_size = df['comment_text'], 1000

    df['comment_text'] = pd.Series(lemmatize(all_texts, batch_size))

    df = quality_check(df)

    return df

In [6]:
def get_processed(path, new_path):
    df = pd.read_csv(path)
    df = text_preprocessing(df)
    df.to_csv(new_path)

In [7]:
train_path = r'../data/raw/train.csv.zip'
test_path = r'../data/raw/test.csv.zip'

new_train = r'../data/processed/train.csv'
new_test = r'../data/processed/test.csv'

In [ ]:
get_processed(train_path, new_train)

Processing text....: 0it [00:00, ?it/s]

In [ ]:
get_processed(test_path, new_test)